In [11]:
import pandas as pd
import requests
import json
from dotenv import load_dotenv
import os
import pandas as pd
import time
from time import sleep
load_dotenv()
API_KEY = os.getenv("API_KEY")

In [16]:
df1 = pd.read_csv("authors_data.csv")

indexes = df1.index[
    (df1["works_count"] >= 5) &
    (df1["works_count"] <= 5000)
].tolist()

subset = df1.loc[indexes, "author_id"]
print(len(subset))

796


In [17]:
# Sociology, Psychology, Economics, Political Science
css_concepts = ["C157449823", "C77805123", "C162324750", "C17744445"]
# Mathematics, Physics, Computer Science
quant_concepts = ["C33923547", "C121332964", "C41008148"]

# Build the concept filter string
# Works must have ANY CSS concept AND ANY quant concept
concept_filter1 = f"concept.id:{'|'.join(css_concepts)}"
concept_filter2 = f"concept.id:{'|'.join(quant_concepts)}"

import requests
import pandas as pd
from itertools import islice
from time import sleep

BASE_URL = "https://api.openalex.org/works"

def chunked(iterable, size):
    it = iter(iterable)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

rows2 = []
rows3 = []
seen_work_ids = set()

for batch_num, author_batch in enumerate(chunked(subset, 25), start=1):

    author_filter = "author.id:" + "|".join(author_batch)

    filter_string = (
    f"{author_filter},"
    f"cited_by_count:>10,"
    f"authors_count:<10,"
    f"{concept_filter1},"
    f"{concept_filter2}"
)

    cursor = "*"

    print(f"\nProcessing batch {batch_num}")

    while cursor:

        params = {
            "filter": filter_string,
            "select": "publication_year,id,cited_by_count,corresponding_author_ids,title,abstract_inverted_index",
            "per-page": 200,
            "cursor": cursor,
            "api_key": API_KEY
        }

        response = requests.get(BASE_URL, params=params)
        data = response.json()

        results = data.get("results", [])
        if not results:
            break

        print(f"  Fetched {len(results)} works")

        for item in results:
            work_id = item.get("id")

            # Deduplicate across batches
            if work_id in seen_work_ids:
                continue
            seen_work_ids.add(work_id)

            rows2.append({
                "id": work_id,
                "publication_year": item.get("publication_year"),
                "cited_by_count": item.get("cited_by_count"),
                "corresponding_author_ids": item.get("corresponding_author_ids"),
            })

            rows3.append({
                "id": work_id,
                "title": item.get("title"),
                "abstract_index": item.get("abstract_inverted_index"),
            })

        cursor = data["meta"].get("next_cursor")

        sleep(0.1)

    sleep(0.5)

df2 = pd.DataFrame(rows2)
df3 = pd.DataFrame(rows3)

print("\nComplete!")
len(df2), len(df3)


Processing batch 1
  Fetched 200 works
  Fetched 131 works

Processing batch 2
  Fetched 200 works
  Fetched 183 works

Processing batch 3
  Fetched 200 works
  Fetched 51 works

Processing batch 4
  Fetched 111 works

Processing batch 5
  Fetched 200 works
  Fetched 115 works

Processing batch 6
  Fetched 200 works
  Fetched 151 works

Processing batch 7
  Fetched 200 works
  Fetched 67 works

Processing batch 8
  Fetched 200 works
  Fetched 200 works
  Fetched 21 works

Processing batch 9
  Fetched 200 works
  Fetched 66 works

Processing batch 10
  Fetched 200 works

Processing batch 11
  Fetched 116 works

Processing batch 12
  Fetched 194 works

Processing batch 13
  Fetched 144 works

Processing batch 14
  Fetched 119 works

Processing batch 15
  Fetched 170 works

Processing batch 16
  Fetched 177 works

Processing batch 17
  Fetched 200 works
  Fetched 150 works

Processing batch 18
  Fetched 200 works
  Fetched 185 works

Processing batch 19
  Fetched 200 works
  Fetched 171 

(7075, 7075)

In [13]:
df2.to_csv("works_ids.csv", index=False)
df3.to_csv("works_titles_abstracts.csv", index=False)

## Co-Authors)

In [15]:
co_author_ids = []
corresponding_unique_author_ids = df2["corresponding_author_ids"].explode().unique()
for author_id in corresponding_unique_author_ids:
    if author_id not in df1["author_id"].values:
        co_author_ids.append(author_id)

len(co_author_ids)

254

In [ ]:
# rows4 = []
# for coauthor_id in co_author_ids:
#     URL = "https://api.openalex.org/authors"

#     params = {
#         "filter": f"id:{coauthor_id}",
#         "select": "display_name,works_count,works_api_url,last_known_institutions,summary_stats",
#         "api_key": API_KEY
#     }

#     response = requests.get(URL, params=params)
#     data = response.json()
    
#     if data.get("results"):
#         data = data["results"][0]
#         h_index = data.get("summary_stats", {}).get("h_index")

#         # Get first institution country code
#         institutions = data.get("last_known_institutions", [])
#         country_code = institutions[0].get("country_code") if institutions else None
#         rows4.append({
#                 "display_name": data.get("display_name"),
#                 "works_count": data.get("works_count"),
#                 "works_api_url": data.get("works_api_url"),
#                 "h_index": h_index,
#                 "country_code": country_code
#             })
    
    
# df4 = pd.DataFrame(rows4)
# df4.to_csv("co_authors_data.csv", index=False)

## Co-author Works

In [ ]:
# rows5 = []
# rows6 = []
# for co_author_id in co_author_ids:
#     # Handle pagination
#     page = 1
#     has_more = True
#     unique_author_ids = set()
    
#     while has_more:
#         # Fixed filter syntax (removed .search)
#         params = {
#             "filter": f"corresponding_author_ids:{co_author_id},cited_by_count:>10,authors_count:<10",
#             "select": "publication_year,id,cited_by_count,corresponding_author_ids,title,abstract_inverted_index",
#             "per-page": 200,
#             "page": page,
#             "api_key": API_KEY
#         }
        
        
#         response = requests.get('https://api.openalex.org/works', params=params)
#         data = response.json()
            
            
            
#         if not data.get("results"):
#             break  # No more results
            
#         results = data["results"]
#         for work_id in results[0].get("id"):
#             if work_id in df2["id"].values:
#                 break
#         else:
#             pass
#         only_co_and_author = []
#         corresponding_author_ids = item.get("corresponding_author_ids")
#         for corresponding_author_id in corresponding_author_ids:
#             if corresponding_author_id in co_author_ids:
#                 only_co_and_author.append(corresponding_author_id)
#             elif corresponding_author_id in subset:
#                 only_co_and_author.append(corresponding_author_id)

            

        
#         for item in results:
#             rows5.append({
#                 "id": item.get('id'),
#                 "publication_year": item.get("publication_year"),
#                 "cited_by_count": item.get("cited_by_count"),
#                 "corresponding_author_ids": only_co_and_author,
#             })
#             rows6.append({
#                 "id": item.get('id'),
#                 "title": item.get('title'),
#                 "abstract_index": item.get('abstract_inverted_index')  
#             })
        
#         # Check if this was the last page
#         if len(results) < 200:
#             has_more = False
#         else:
#             sleep(0.1)

# # Be nice to the API - delay between authors
#     sleep(0.5)

# # Create DataFrames
# df5 = pd.DataFrame(rows5)
# df6 = pd.DataFrame(rows6)

# print(f"\nComplete!")
# print(f"df5: {len(df5)} rows")
# print(f"df6: {len(df6)} rows")



Complete!
df5: 1168380 rows
df6: 1168380 rows


In [66]:
df5.to_csv("co_authors_works_ids.csv", index=False)
df6.to_csv("co_authors_works_titles_abstracts.csv", index=False)